In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = (
    Path("../data/processed")
    / "cic_ids2017_unified.csv"
)

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

Shape: (2574264, 79)


In [3]:
for i, column in enumerate(df.columns):
    print(f"{i:02d} -> {column}")

00 -> Destination_Port
01 -> Flow_Duration
02 -> Total_Fwd_Packets
03 -> Total_Backward_Packets
04 -> Total_Length_of_Fwd_Packets
05 -> Total_Length_of_Bwd_Packets
06 -> Fwd_Packet_Length_Max
07 -> Fwd_Packet_Length_Min
08 -> Fwd_Packet_Length_Mean
09 -> Fwd_Packet_Length_Std
10 -> Bwd_Packet_Length_Max
11 -> Bwd_Packet_Length_Min
12 -> Bwd_Packet_Length_Mean
13 -> Bwd_Packet_Length_Std
14 -> Flow_Bytes_per_s
15 -> Flow_Packets_per_s
16 -> Flow_IAT_Mean
17 -> Flow_IAT_Std
18 -> Flow_IAT_Max
19 -> Flow_IAT_Min
20 -> Fwd_IAT_Total
21 -> Fwd_IAT_Mean
22 -> Fwd_IAT_Std
23 -> Fwd_IAT_Max
24 -> Fwd_IAT_Min
25 -> Bwd_IAT_Total
26 -> Bwd_IAT_Mean
27 -> Bwd_IAT_Std
28 -> Bwd_IAT_Max
29 -> Bwd_IAT_Min
30 -> Fwd_PSH_Flags
31 -> Bwd_PSH_Flags
32 -> Fwd_URG_Flags
33 -> Bwd_URG_Flags
34 -> Fwd_Header_Length
35 -> Bwd_Header_Length
36 -> Fwd_Packets_per_s
37 -> Bwd_Packets_per_s
38 -> Min_Packet_Length
39 -> Max_Packet_Length
40 -> Packet_Length_Mean
41 -> Packet_Length_Std
42 -> Packet_Length_Varian

In [4]:
for column in df.columns:

    name = column.lower()

    if any(
        keyword in name
        for keyword in [
            "ip",
            "timestamp",
            "port",
            "protocol"
        ]
    ):
        print(column)

Destination_Port


In [5]:
THREAT_INTELLIGENCE_COLUMNS = []

ML_EXCLUDED_COLUMNS = []

for column in df.columns:

    name = column.lower()

    if any(
        keyword in name
        for keyword in [
            "source_ip",
            "destination_ip",
            "timestamp"
        ]
    ):
        THREAT_INTELLIGENCE_COLUMNS.append(column)

print("Threat intelligence / contextual columns:")

for column in THREAT_INTELLIGENCE_COLUMNS:
    print("-", column)

Threat intelligence / contextual columns:


In [6]:
NETWORK_CONTEXT_COLUMNS = []

for column in df.columns:

    name = column.lower()

    if any(
        keyword in name
        for keyword in [
            "port",
            "protocol"
        ]
    ):
        NETWORK_CONTEXT_COLUMNS.append(column)

print("\nNetwork context columns:")

for column in NETWORK_CONTEXT_COLUMNS:
    print("-", column)


Network context columns:
- Destination_Port


In [7]:
TARGET_COLUMN = "Label"

print("Target column:", TARGET_COLUMN)

Target column: Label


In [8]:
print(df[TARGET_COLUMN].value_counts())

Label
BENIGN                        2148386
DoS Hulk                       172849
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [9]:
constant_columns = []

for column in df.columns:

    if df[column].nunique(dropna=False) <= 1:
        constant_columns.append(column)

print("Constant columns:")

for column in constant_columns:
    print("-", column)

Constant columns:
- Bwd_PSH_Flags
- Bwd_URG_Flags
- Fwd_Avg_Bytes_per_Bulk
- Fwd_Avg_Packets_per_Bulk
- Fwd_Avg_Bulk_Rate
- Bwd_Avg_Bytes_per_Bulk
- Bwd_Avg_Packets_per_Bulk
- Bwd_Avg_Bulk_Rate


In [10]:
unique_ratio = (
    df.nunique(dropna=False) / len(df)
)

high_cardinality = (
    unique_ratio[unique_ratio > 0.95]
    .sort_values(ascending=False)
)

print("High-cardinality columns:")
print(high_cardinality)

High-cardinality columns:
Series([], dtype: float64)


In [11]:
numeric_df = df.select_dtypes(
    include=np.number
)

correlation = numeric_df.corr()

print(
    correlation.abs()
    .stack()
    .sort_values(ascending=False)
    .head(30)
)

Bwd_IAT_Total                Bwd_IAT_Total                  1.0
Idle_Min                     Idle_Min                       1.0
Fwd_IAT_Min                  Fwd_IAT_Min                    1.0
ACK_Flag_Count               ACK_Flag_Count                 1.0
Bwd_Packet_Length_Max        Bwd_Packet_Length_Max          1.0
Idle_Max                     Idle_Max                       1.0
Subflow_Bwd_Bytes            Subflow_Bwd_Bytes              1.0
Flow_Duration                Flow_Duration                  1.0
Avg_Bwd_Segment_Size         Bwd_Packet_Length_Mean         1.0
Fwd_IAT_Max                  Fwd_IAT_Max                    1.0
SYN_Flag_Count               Fwd_PSH_Flags                  1.0
FIN_Flag_Count               FIN_Flag_Count                 1.0
Bwd_Packet_Length_Std        Bwd_Packet_Length_Std          1.0
Init_Win_bytes_backward      Init_Win_bytes_backward        1.0
Total_Length_of_Bwd_Packets  Total_Length_of_Bwd_Packets    1.0
act_data_pkt_fwd             act_data_pk

In [12]:
duplicate_columns = []

columns = df.columns

for i in range(len(columns)):

    for j in range(i + 1, len(columns)):

        if df[columns[i]].equals(
            df[columns[j]]
        ):
            duplicate_columns.append(
                (columns[i], columns[j])
            )

print("Duplicate column pairs:")

for pair in duplicate_columns:
    print(pair)

Duplicate column pairs:
('Total_Fwd_Packets', 'Subflow_Fwd_Packets')
('Total_Backward_Packets', 'Subflow_Bwd_Packets')
('Fwd_PSH_Flags', 'SYN_Flag_Count')
('Bwd_PSH_Flags', 'Bwd_URG_Flags')
('Bwd_PSH_Flags', 'Fwd_Avg_Bytes_per_Bulk')
('Bwd_PSH_Flags', 'Fwd_Avg_Packets_per_Bulk')
('Bwd_PSH_Flags', 'Fwd_Avg_Bulk_Rate')
('Bwd_PSH_Flags', 'Bwd_Avg_Bytes_per_Bulk')
('Bwd_PSH_Flags', 'Bwd_Avg_Packets_per_Bulk')
('Bwd_PSH_Flags', 'Bwd_Avg_Bulk_Rate')
('Fwd_URG_Flags', 'CWE_Flag_Count')
('Bwd_URG_Flags', 'Fwd_Avg_Bytes_per_Bulk')
('Bwd_URG_Flags', 'Fwd_Avg_Packets_per_Bulk')
('Bwd_URG_Flags', 'Fwd_Avg_Bulk_Rate')
('Bwd_URG_Flags', 'Bwd_Avg_Bytes_per_Bulk')
('Bwd_URG_Flags', 'Bwd_Avg_Packets_per_Bulk')
('Bwd_URG_Flags', 'Bwd_Avg_Bulk_Rate')
('Fwd_Header_Length', 'Fwd_Header_Length.1')
('Fwd_Avg_Bytes_per_Bulk', 'Fwd_Avg_Packets_per_Bulk')
('Fwd_Avg_Bytes_per_Bulk', 'Fwd_Avg_Bulk_Rate')
('Fwd_Avg_Bytes_per_Bulk', 'Bwd_Avg_Bytes_per_Bulk')
('Fwd_Avg_Bytes_per_Bulk', 'Bwd_Avg_Packets_per_Bulk')
('

In [13]:
EXCLUDED_FROM_ML = []

for column in df.columns:

    name = column.lower()

    if column == TARGET_COLUMN:
        continue

    if any(
        keyword in name
        for keyword in [
            "source_ip",
            "destination_ip",
            "timestamp"
        ]
    ):
        EXCLUDED_FROM_ML.append(column)

ML_FEATURES = [
    column
    for column in df.columns
    if column not in EXCLUDED_FROM_ML
    and column != TARGET_COLUMN
]

print("Total columns:", len(df.columns))
print("Excluded from ML:", len(EXCLUDED_FROM_ML))
print("Initial ML features:", len(ML_FEATURES))

Total columns: 79
Excluded from ML: 0
Initial ML features: 78


In [14]:
print("\nExcluded columns:")

for column in EXCLUDED_FROM_ML:
    print("-", column)


Excluded columns:


In [15]:
print("\nInitial ML features:")

for column in ML_FEATURES:
    print("-", column)


Initial ML features:
- Destination_Port
- Flow_Duration
- Total_Fwd_Packets
- Total_Backward_Packets
- Total_Length_of_Fwd_Packets
- Total_Length_of_Bwd_Packets
- Fwd_Packet_Length_Max
- Fwd_Packet_Length_Min
- Fwd_Packet_Length_Mean
- Fwd_Packet_Length_Std
- Bwd_Packet_Length_Max
- Bwd_Packet_Length_Min
- Bwd_Packet_Length_Mean
- Bwd_Packet_Length_Std
- Flow_Bytes_per_s
- Flow_Packets_per_s
- Flow_IAT_Mean
- Flow_IAT_Std
- Flow_IAT_Max
- Flow_IAT_Min
- Fwd_IAT_Total
- Fwd_IAT_Mean
- Fwd_IAT_Std
- Fwd_IAT_Max
- Fwd_IAT_Min
- Bwd_IAT_Total
- Bwd_IAT_Mean
- Bwd_IAT_Std
- Bwd_IAT_Max
- Bwd_IAT_Min
- Fwd_PSH_Flags
- Bwd_PSH_Flags
- Fwd_URG_Flags
- Bwd_URG_Flags
- Fwd_Header_Length
- Bwd_Header_Length
- Fwd_Packets_per_s
- Bwd_Packets_per_s
- Min_Packet_Length
- Max_Packet_Length
- Packet_Length_Mean
- Packet_Length_Std
- Packet_Length_Variance
- FIN_Flag_Count
- SYN_Flag_Count
- RST_Flag_Count
- PSH_Flag_Count
- ACK_Flag_Count
- URG_Flag_Count
- CWE_Flag_Count
- ECE_Flag_Count
- Down_per_

In [ ]:
report_path = Path(
    "../reports/feature_audit.txt"
)

with open(report_path, "w") as f:

    f.write(
        "CIC-IDS2017 Feature Audit\n"
    )

    f.write("=" * 50 + "\n\n")

    f.write(
        f"Total columns: {len(df.columns)}\n"
    )

    f.write(
        f"Total ML features: {len(ML_FEATURES)}\n"
    )

    f.write(
        f"Excluded from ML: "
        f"{len(EXCLUDED_FROM_ML)}\n\n"
    )

    f.write(
        "Excluded Columns\n"
    )

    f.write("-" * 50 + "\n")

    for column in EXCLUDED_FROM_ML:
        f.write(f"{column}\n")

    f.write(
        "\nML Features\n"
    )

    f.write("-" * 50 + "\n")

    for column in ML_FEATURES:
        f.write(f"{column}\n")

print(
    f"Report saved to: {report_path}"
)

Report saved to: ..\reports\feature_audit.txt


: 